Viscous Flux Jacobian Formulation in 3D
- projected in normal unit vector n = [nx,ny,nz]T
- two possible formulations : by analytical method and by finite difference method

In [10]:
"""
3D Normal Viscous Flux Jacobian (interface between two control volumes)
========================================================================

Implements d(F_n^v)/dU_i  and  d(F_n^v)/dU_j, i.e. the Jacobian of the
normal (projected) viscous flux F_n^v with respect to the conservative
variables of the two cells (i = "left", j = "right") sharing a face,
following the construction:

        F_n^v = [0, -tau_nx, -tau_ny, -tau_nz, -tau_nn + q_n]^T      (eq. 4.12.5)

        d F_n^v / dU = (d F_n^v / dW) * (dW / dU)                    (eq. 4.12.15)

with W = [rho, u, v, w, p]^T.

Modeling choices (consistent with the reference derivation):
  * Interface (averaged) primitive states are the simple arithmetic
    mean of the two cells:           phi_avg = (phi_i + phi_j) / 2   (eq. 4.12.25)
  * The normal derivative of any quantity is the directional
    finite difference across the two cell centers:
            d(phi)/dn = (phi_j - phi_i) / ds                          (eq. 4.12.26)
  * Because only the normal derivative is available (no compact
    stencil for the tangential derivatives), the full gradient is
    approximated as  grad(phi) ~= (d phi/dn) * n   (i.e. only the
    derivative along n is retained -- the same simplification used
    to reduce tau_bar.n to a function of d()/dn only, eq. 4.12.12-14).
  * Viscosity follows Sutherland's law (eq. 4.4.4) and its dependence
    on rho and p (through T = p/(rho*R)) is differentiated exactly
    via the symbolic chain rule -- this reproduces eq. (4.12.20)-(4.12.24)
    without having to hand-transcribe each partial derivative.

The two Jacobian blocks (wrt U_i and wrt U_j) are exactly what an
implicit (e.g. Newton/line-implicit/LU-SGS) viscous-flux assembly
needs for the off-diagonal/diagonal contributions of a face.

Author: generated to accompany the "3D Normal Viscous Flux and Jacobian"
notes (section 4.12).
"""

import numpy as np
import sympy as sp
import time


# ----------------------------------------------------------------------
# Conservative -> primitive variables
# ----------------------------------------------------------------------
def cons_to_prim(U, gamma, R_gas):
    """
    U = [rho, rho*u, rho*v, rho*w, rho*E]
    returns rho, u, v, w, p, T  (numeric floats)
    """
    rho = U[0]
    u = U[1] / rho
    v = U[2] / rho
    w = U[3] / rho
    E = U[4] / rho
    p = (gamma - 1.0) * rho * (E - 0.5 * (u * u + v * v + w * w))
    T = p / (rho * R_gas)
    return rho, u, v, w, p, T


# ----------------------------------------------------------------------
# Sutherland's law (eq. 4.4.4), symbolic-friendly
# ----------------------------------------------------------------------
def sutherland_mu(T, mu0, T0, C):
    return mu0 * (T0 + C) / (T + C) * (T / T0) ** sp.Rational(3, 2)


# ----------------------------------------------------------------------
# dW/dU  (the right-hand matrix of eq. 4.12.15), evaluated at a given
# primitive state.  W = [rho, u, v, w, p]^T , U = [rho, rho u, rho v, rho w, rho E]^T
# ----------------------------------------------------------------------
def dWdU(rho, u, v, w, p, gamma):
    q2 = u * u + v * v + w * w
    M = np.array([
        [1.0,                 0.0,            0.0,            0.0,            0.0],
        [-u / rho,            1.0 / rho,       0.0,            0.0,            0.0],
        [-v / rho,            0.0,             1.0 / rho,      0.0,            0.0],
        [-w / rho,            0.0,             0.0,            1.0 / rho,      0.0],
        [0.5 * (gamma - 1.0) * q2, -(gamma - 1.0) * u, -(gamma - 1.0) * v, -(gamma - 1.0) * w, (gamma - 1.0)]
    ])
    return M


# ----------------------------------------------------------------------
# Symbolic construction of F_n^v as a function of the LEFT state
# (rho_L,u_L,v_L,w_L,p_L) and the RIGHT state (rho_R,u_R,v_R,w_R,p_R).
# Works for either state being symbolic; the other is plugged in as
# plain numbers.
# ----------------------------------------------------------------------
def _Fnv_symbolic(rho_L, u_L, v_L, w_L, p_L,
                   rho_R, u_R, v_R, w_R, p_R,
                   nx, ny, nz, ds,
                   gamma, R_gas, Pr, mu0, T0, Suth_C):

    T_L = p_L / (rho_L * R_gas)
    T_R = p_R / (rho_R * R_gas)

    # interface-averaged primitive state                       (eq. 4.12.25 style)
    rho_avg = (rho_L + rho_R) / 2
    u_avg   = (u_L   + u_R)   / 2
    v_avg   = (v_L   + v_R)   / 2
    w_avg   = (w_L   + w_R)   / 2
    T_avg   = (T_L   + T_R)   / 2

    mu_avg = sutherland_mu(T_avg, mu0, T0, Suth_C)             # mu(T_avg(rho,p)) -> exact chain rule via sympy

    # normal derivatives                                        (eq. 4.12.26)
    dudn = (u_R - u_L) / ds
    dvdn = (v_R - v_L) / ds
    dwdn = (w_R - w_L) / ds
    dTdn = (T_R - T_L) / ds

    # full gradient approximated by the normal derivative only:
    #   grad(phi) ~= (d phi/dn) * n   =>   d phi/dx_k = (d phi/dn) * n_k
    div = dudn * nx + dvdn * ny + dwdn * nz   # du/dx + dv/dy + dw/dz

    # Newtonian viscous stress tensor with Stokes' hypothesis      (eq. 4.12.16-19)
    tau_xx = mu_avg * (2 * dudn * nx - sp.Rational(2, 3) * div)
    tau_yy = mu_avg * (2 * dvdn * ny - sp.Rational(2, 3) * div)
    tau_zz = mu_avg * (2 * dwdn * nz - sp.Rational(2, 3) * div)
    tau_xy = mu_avg * (dudn * ny + dvdn * nx)
    tau_xz = mu_avg * (dudn * nz + dwdn * nx)
    tau_yz = mu_avg * (dvdn * nz + dwdn * ny)

    # projection along n                                          (eq. 4.12.6-4.12.9)
    tau_nx = tau_xx * nx + tau_xy * ny + tau_xz * nz
    tau_ny = tau_xy * nx + tau_yy * ny + tau_yz * nz
    tau_nz = tau_xz * nx + tau_yz * ny + tau_zz * nz
    tau_nn = tau_nx * u_avg + tau_ny * v_avg + tau_nz * w_avg

    # heat flux                                                    (eq. 4.12.11)
    kappa_avg = gamma * mu_avg / (Pr * (gamma - 1.0))
    q_n = -kappa_avg * dTdn

    F0 = sp.Integer(0)
    F1 = -tau_nx
    F2 = -tau_ny
    F3 = -tau_nz
    F4 = -tau_nn + q_n
    return sp.Matrix([F0, F1, F2, F3, F4])


# ----------------------------------------------------------------------
# Purely numeric evaluation of F_n^v(U_i, U_j, n, ds)  -- the discretized
# normal viscous flux function itself (no differentiation). This is the
# "flux(U)" used by the finite-difference Jacobian below, and it uses
# exactly the same modeling choices as the analytic version above:
#   * interface state = arithmetic mean of the two cells   (eq. 4.12.25)
#   * spatial (normal) derivative = directional finite difference
#         d(phi)/dn = (phi_j - phi_i) / ds                  (eq. 4.12.26)
#     i.e. the ONLY spatial derivative information available is the
#     one-sided difference of the two cell-center values along the
#     line joining them; the gradient is then assumed aligned with n:
#         grad(phi) ~= (d phi/dn) * n
#     (tangential derivatives are not reconstructed/are neglected).
# ----------------------------------------------------------------------
def Fnv_numeric(U_i, U_j, n, ds,
                 gamma=1.4, R_gas=287.0, Pr=0.72,
                 mu0=1.716e-5, T0=273.15, Suth_C=110.4):
    nx, ny, nz = n
    rho_L, u_L, v_L, w_L, p_L, T_L = cons_to_prim(U_i, gamma, R_gas)
    rho_R, u_R, v_R, w_R, p_R, T_R = cons_to_prim(U_j, gamma, R_gas)

    u_avg = (u_L + u_R) / 2
    v_avg = (v_L + v_R) / 2
    w_avg = (w_L + w_R) / 2
    T_avg = (T_L + T_R) / 2

    mu_avg = mu0 * (T0 + Suth_C) / (T_avg + Suth_C) * (T_avg / T0) ** 1.5

    dudn = (u_R - u_L) / ds
    dvdn = (v_R - v_L) / ds
    dwdn = (w_R - w_L) / ds
    dTdn = (T_R - T_L) / ds

    div = dudn * nx + dvdn * ny + dwdn * nz

    tau_xx = mu_avg * (2 * dudn * nx - (2.0 / 3.0) * div)
    tau_yy = mu_avg * (2 * dvdn * ny - (2.0 / 3.0) * div)
    tau_zz = mu_avg * (2 * dwdn * nz - (2.0 / 3.0) * div)
    tau_xy = mu_avg * (dudn * ny + dvdn * nx)
    tau_xz = mu_avg * (dudn * nz + dwdn * nx)
    tau_yz = mu_avg * (dvdn * nz + dwdn * ny)

    tau_nx = tau_xx * nx + tau_xy * ny + tau_xz * nz
    tau_ny = tau_xy * nx + tau_yy * ny + tau_yz * nz
    tau_nz = tau_xz * nx + tau_yz * ny + tau_zz * nz
    tau_nn = tau_nx * u_avg + tau_ny * v_avg + tau_nz * w_avg

    kappa_avg = gamma * mu_avg / (Pr * (gamma - 1.0))
    q_n = -kappa_avg * dTdn

    return np.array([0.0, -tau_nx, -tau_ny, -tau_nz, -tau_nn + q_n])


# ----------------------------------------------------------------------
# Finite-difference Jacobian (separate, independent function)
# ----------------------------------------------------------------------
def viscous_flux_jacobians_fd(U_i, U_j, n, ds, eps,
                               gamma=1.4, R_gas=287.0, Pr=0.72,
                               mu0=1.716e-5, T0=273.15, Suth_C=110.4):
    """
    Finite-difference estimate of d(F_n^v)/d(U_i) and d(F_n^v)/d(U_j),
    using the SAME discretized flux function F_n^v(U_i, U_j, n, ds) as
    the analytic version (Fnv_numeric above), but differentiating it
    by brute-force one-sided (forward) perturbation instead of exact
    symbolic/analytic differentiation.

    How the spatial derivative is determined
    -----------------------------------------
    The flux function F_n^v(U_i, U_j, n, ds) itself still needs a
    normal *spatial* gradient at the face. That gradient is built
    exactly as in the analytic implementation:
        d(phi)/dn = (phi_j - phi_i) / ds          (one-sided/directional
                                                    finite difference
                                                    across the two cell
                                                    centers, eq. 4.12.26)
        grad(phi) ~= (d phi/dn) * n                (tangential gradient
                                                    components are not
                                                    available/neglected;
                                                    eq. 4.12.12-14)
    This part is NOT what "eps" controls -- it is fixed by the mesh
    geometry (n, ds) and is identical to the analytic Jacobian's
    assumption.

    How the JACOBIAN (d Flux/d U) is determined
    ---------------------------------------------
    "eps" is a separate, independent perturbation used only to
    differentiate the flux function with respect to the conservative
    variables, via a forward finite difference, one component at a
    time:

        d F_n^v / dU_i[:,k]  ~=  ( F_n^v(U_i + eps*e_k, U_j, n, ds)
                                    - F_n^v(U_i, U_j, n, ds) ) / eps

        d F_n^v / dU_j[:,k]  ~=  ( F_n^v(U_i, U_j + eps*e_k, n, ds)
                                    - F_n^v(U_i, U_j, n, ds) ) / eps

    where e_k is the k-th unit vector in conservative-variable space
    (rho, rho*u, rho*v, rho*w, rho*E) and eps is a small, user-specified
    ABSOLUTE perturbation added directly to that conservative variable
    (same units as U_i[k]/U_j[k]). This is a forward difference, first
    order accurate in eps, i.e. error ~ O(eps); the other cell's state,
    n and ds are held fixed during each perturbation.

    Note: the formula is divided by eps (the perturbation), NOT by U;
    dividing by U would not produce a derivative -- it is eps that
    plays the role of "delta U" in (Flux(U+eps) - Flux(U)) / eps.

    Parameters
    ----------
    U_i, U_j : array_like, shape (5,)
        Conservative variables of the two cells sharing the face.
    n : array_like, shape (3,)
        Unit normal vector [nx, ny, nz], pointing from i to j.
    ds : float
        Distance between the two cell centers.
    eps : float
        User-specified absolute perturbation applied to each
        conservative-variable component in turn.
    gamma, R_gas, Pr, mu0, T0, Suth_C :
        Same gas/transport parameters as the analytic version.

    Returns
    -------
    J_i_fd : ndarray, shape (5,5)
        Finite-difference estimate of d(F_n^v)/d(U_i).
    J_j_fd : ndarray, shape (5,5)
        Finite-difference estimate of d(F_n^v)/d(U_j).
    """
    U_i = np.asarray(U_i, dtype=float)
    U_j = np.asarray(U_j, dtype=float)

    F_base = Fnv_numeric(U_i, U_j, n, ds, gamma, R_gas, Pr, mu0, T0, Suth_C)

    J_i_fd = np.zeros((5, 5))
    J_j_fd = np.zeros((5, 5))

    for k in range(5):
        # --- perturb cell i, component k ---
        U_i_pert = U_i.copy()
        U_i_pert[k] += eps
        F_pert_i = Fnv_numeric(U_i_pert, U_j, n, ds, gamma, R_gas, Pr, mu0, T0, Suth_C)
        J_i_fd[:, k] = (F_pert_i - F_base) / eps

        # --- perturb cell j, component k ---
        U_j_pert = U_j.copy()
        U_j_pert[k] += eps
        F_pert_j = Fnv_numeric(U_i, U_j_pert, n, ds, gamma, R_gas, Pr, mu0, T0, Suth_C)
        J_j_fd[:, k] = (F_pert_j - F_base) / eps

    return J_i_fd, J_j_fd


# ----------------------------------------------------------------------
# Main entry point
# ----------------------------------------------------------------------
def viscous_flux_jacobians(U_i, U_j, n, ds,
                            gamma=1.4, R_gas=287.0, Pr=0.72,
                            mu0=1.716e-5, T0=273.15, Suth_C=110.4):
    """
    Compute the normal viscous flux Jacobian blocks at a cell face between
    cell i (left/owner) and cell j (right/neighbor).

    Parameters
    ----------
    U_i, U_j : array_like, shape (5,)
        Conservative variables [rho, rho*u, rho*v, rho*w, rho*E] of the
        two cells sharing the face.
    n : array_like, shape (3,)
        Unit normal vector [nx, ny, nz] of the face, pointing from i to j.
    ds : float
        Distance between the two cell centers (used for the directional
        finite-difference normal derivative, eq. 4.12.26).
    gamma, R_gas, Pr : float
        Ratio of specific heats, specific gas constant, Prandtl number.
    mu0, T0, Suth_C : float
        Sutherland's law reference viscosity, reference temperature and
        Sutherland constant (eq. 4.4.4).

    Returns
    -------
    J_i : ndarray, shape (5,5)
        d(F_n^v)/d(U_i)  -- Jacobian wrt the left-cell conservative state.
    J_j : ndarray, shape (5,5)
        d(F_n^v)/d(U_j)  -- Jacobian wrt the right-cell conservative state.
    """
    U_i = np.asarray(U_i, dtype=float)
    U_j = np.asarray(U_j, dtype=float)
    nx, ny, nz = n

    rho_i, u_i, v_i, w_i, p_i, _ = cons_to_prim(U_i, gamma, R_gas)
    rho_j, u_j, v_j, w_j, p_j, _ = cons_to_prim(U_j, gamma, R_gas)

    rho_s, u_s, v_s, w_s, p_s = sp.symbols('rho_s u_s v_s w_s p_s')
    Wsyms = (rho_s, u_s, v_s, w_s, p_s)

    # ---- Jacobian wrt the LEFT state (U_i) ----
    Fnv_i = _Fnv_symbolic(rho_s, u_s, v_s, w_s, p_s,
                           rho_j, u_j, v_j, w_j, p_j,
                           nx, ny, nz, ds, gamma, R_gas, Pr, mu0, T0, Suth_C)
    dFdW_i = Fnv_i.jacobian(Wsyms)
    dFdW_i_num = np.array(
        dFdW_i.subs({rho_s: rho_i, u_s: u_i, v_s: v_i, w_s: w_i, p_s: p_i})
    ).astype(np.float64)
    J_i = dFdW_i_num @ dWdU(rho_i, u_i, v_i, w_i, p_i, gamma)

    # ---- Jacobian wrt the RIGHT state (U_j) ----
    Fnv_j = _Fnv_symbolic(rho_i, u_i, v_i, w_i, p_i,
                           rho_s, u_s, v_s, w_s, p_s,
                           nx, ny, nz, ds, gamma, R_gas, Pr, mu0, T0, Suth_C)
    dFdW_j = Fnv_j.jacobian(Wsyms)
    dFdW_j_num = np.array(
        dFdW_j.subs({rho_s: rho_j, u_s: u_j, v_s: v_j, w_s: w_j, p_s: p_j})
    ).astype(np.float64)
    J_j = dFdW_j_num @ dWdU(rho_j, u_j, v_j, w_j, p_j, gamma)

    return J_i, J_j


# ----------------------------------------------------------------------
# Example / sanity check
# ----------------------------------------------------------------------
if __name__ == "__main__":
    np.set_printoptions(precision=4, suppress=False, linewidth=120)

    gamma = 1.4
    R_gas = 287.0

    # Build two arbitrary, physically reasonable conservative states
    def prim_to_cons(rho, u, v, w, p, gamma):
        E = p / ((gamma - 1.0) * rho) + 0.5 * (u * u + v * v + w * w)
        return np.array([rho, rho * u, rho * v, rho * w, rho * E])

    U_i = prim_to_cons(rho=1.4e-6, u=50.0, v=2.0, w=0.0, p=1.35, gamma=gamma)
    U_j = prim_to_cons(rho=1.6e-6, u=55.0, v=1.5, w=0.5, p=1.45, gamma=gamma)

    n = np.array([1.0, 0.0, 0.0])   # face normal pointing from i to j
    ds = 0.01                       # distance between cell centers [m]

    J_i, J_j = viscous_flux_jacobians(U_i, U_j, n, ds, gamma=gamma, R_gas=R_gas)

    print("ANALYTIC  d(F_n^v)/d(U_i):")
    print(J_i)
    print()
    print("ANALYTIC  d(F_n^v)/d(U_j):")
    print(J_j)

    # ---- finite-difference check ----
    eps = 1.0e-8  # user-specified absolute perturbation (same units as U)
    J_i_fd, J_j_fd = viscous_flux_jacobians_fd(U_i, U_j, n, ds, eps, gamma=gamma, R_gas=R_gas)

    print()
    print("FINITE-DIFFERENCE  d(F_n^v)/d(U_i)  (eps = %.1e):" % eps)
    print(J_i_fd)
    print()
    print("FINITE-DIFFERENCE  d(F_n^v)/d(U_j)  (eps = %.1e):" % eps)
    print(J_j_fd)

    print()
    print("max |J_i - J_i_fd| =", np.max(np.abs(J_i - J_i_fd)))
    print("max |J_j - J_j_fd| =", np.max(np.abs(J_j - J_j_fd)))

ANALYTIC  d(F_n^v)/d(U_i):
[[ 0.0000e+00  0.0000e+00  0.0000e+00  0.0000e+00  0.0000e+00]
 [-3.7281e+05  7.6669e+03  8.7345e-03  0.0000e+00 -4.3672e-03]
 [-1.2289e+04 -1.6377e-02  5.7500e+03  0.0000e+00  3.2754e-04]
 [ 7.8920e+02  1.6377e-02  6.5509e-04  5.7500e+03 -3.2754e-04]
 [-1.1405e+08  3.8136e+05  1.1421e+04  0.0000e+00  3.9372e+01]]

ANALYTIC  d(F_n^v)/d(U_j):
[[ 0.0000e+00  0.0000e+00  0.0000e+00  0.0000e+00  0.0000e+00]
 [ 3.7761e+05 -6.7081e+03  5.7320e-03  1.9107e-03 -3.8213e-03]
 [ 6.8980e+03 -1.5763e-02 -5.0312e+03 -1.4330e-04  2.8660e-04]
 [ 3.1645e+03  1.5763e-02  4.2990e-04 -5.0312e+03 -2.8660e-04]
 [ 9.6660e+07 -3.6710e+05 -7.4963e+03 -2.4988e+03 -3.3724e+01]]

FINITE-DIFFERENCE  d(F_n^v)/d(U_i)  (eps = 1.0e-08):
[[ 0.0000e+00  0.0000e+00  0.0000e+00  0.0000e+00  0.0000e+00]
 [-3.6941e+05  7.6669e+03  8.7501e-03  1.5597e-05 -4.3672e-03]
 [-1.2181e+04 -1.6378e-02  5.7500e+03 -1.1698e-06  3.2754e-04]
 [ 7.8437e+02  1.6378e-02  6.5626e-04  5.7500e+03 -3.2754e-04]
 [-1.12

Sweeping different eps to find the optimal value

In [11]:
# ── eps sweep: find optimal FD step ──────────────────────────────────────────
print("\n" + "="*65)
print("eps sweep  (analytic vs FD Jacobian)")
print(f"{'eps':>12}  {'max |J_i - J_i_fd|':>18}  {'max |J_j - J_j_fd|':>18}  {'rel err':>10}")
print("-"*65)

eps_values = [10**(-e) for e in range(5, 20)]   # 1e-5 … 1e-10
results    = []

for eps in eps_values:
    J_i_fd, J_j_fd = viscous_flux_jacobians_fd(
        U_i, U_j, n, ds, eps, gamma=gamma, R_gas=R_gas)

    err_i = np.max(np.abs(J_i - J_i_fd))
    err_j = np.max(np.abs(J_j - J_j_fd))

    # relative error: norm of error / norm of analytic
    rel = (np.linalg.norm(J_i - J_i_fd) + np.linalg.norm(J_j - J_j_fd)) / \
          (np.linalg.norm(J_i) + np.linalg.norm(J_j) + 1e-14)

    results.append((eps, err_i, err_j, rel))
    print(f"{eps:>12.1e}  {err_i:>18.4e}  {err_j:>18.4e}  {rel:>10.4e}")

# ── identify optimal eps (minimum relative error) ─────────────────────────────
best = min(results, key=lambda r: r[3])
print("-"*65)
print(f"Optimal eps: {best[0]:.1e}  →  rel err = {best[3]:.4e}")


eps sweep  (analytic vs FD Jacobian)
         eps  max |J_i - J_i_fd|  max |J_j - J_j_fd|     rel err
-----------------------------------------------------------------
     1.0e-05          1.0465e+08          8.7660e+07  9.1264e-01
     1.0e-06          5.7550e+07          4.5622e+07  4.8963e-01
     1.0e-07          1.0127e+07          7.6452e+06  8.4342e-02
     1.0e-08          1.0938e+06          8.1861e+05  9.0757e-03
     1.0e-09          1.1026e+05          8.2442e+04  9.1450e-04
     1.0e-10          1.1035e+04          8.2500e+03  9.1520e-05
     1.0e-11          1.1035e+03          8.2506e+02  9.1527e-06
     1.0e-12          1.1039e+02          8.2506e+01  9.1543e-07
     1.0e-13          1.1408e+01          8.3683e+00  9.3882e-08
     1.0e-14          2.2869e+00          2.4456e+00  2.5315e-08
     1.0e-15          2.3807e+01          2.7789e+01  3.6024e-07
     1.0e-16          1.6695e+02          1.0047e+02  1.5878e-06
     1.0e-17          3.2784e+03          1.5288e+0

Check the improvement in run time for finite differenncing
- Assume the function is called for N times while assembling the global flux jacobian
- N = Number of cells * 5

In [12]:
# check the improvement in run time for finite differenncing

gamma = 1.4
U_i = prim_to_cons(rho=1.20, u=50.0, v=2.0, w=0.0, p=101325.0, gamma=gamma)
U_j = prim_to_cons(rho=1.18, u=55.0, v=1.5, w=0.5, p=101000.0, gamma=gamma)
n = np.array([1.0, 0.0, 0.0])   # face normal pointing from i to j
ds = 0.01                       # distance between cell centers [m]

eps = 1e-8
N   = 1000   # number of repeated calls (e.g if cells = 3060, called = 3060 * 5 ~ 20000)

# ── Analytical ──────────────────────────────────────────────────────
t0 = time.perf_counter()
for _ in range(N):
    viscous_flux_jacobians(U_i, U_j, n, ds, gamma=gamma, R_gas=R_gas)
t_analytical = (time.perf_counter() - t0) / N * 1e6   # µs per call

# ── Finite difference ───────────────────────────────────────────────
t0 = time.perf_counter()
for _ in range(N):
    J_i_fd, J_j_fd = viscous_flux_jacobians_fd(U_i, U_j, n, ds, eps, gamma=gamma, R_gas=R_gas)
t_fd = (time.perf_counter() - t0) / N * 1e6           # µs per call

print(f"Number of calls N: {N}")
print(f"Analytical  : {t_analytical:.2f} µs/call")
print(f"FD          : {t_fd:.2f} µs/call")
print(f"Ratio of run time (FD/Analytical)    : {t_fd / t_analytical:.3f}")

Number of calls N: 1000
Analytical  : 15007.35 µs/call
FD          : 125.65 µs/call
Ratio of run time (FD/Analytical)    : 0.008
